Static testing via webcam

In [2]:

from keras.models import load_model
import pickle

model = load_model("models/static_fsl_model.keras")


In [3]:
import mediapipe as mp

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=False,  # webcam = dynamic
    max_num_hands=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)
mp_draw = mp.solutions.drawing_utils


In [4]:
import cv2
cap = cv2.VideoCapture(0)  # 0 = default camera


In [5]:
import numpy as np
import pickle
from fsl_preprocessing import normalize_landmarks, preprocess_sample, extract_keypoints_from_hand_landmarks


# -------------------------------
# 1. Load artifacts
# -------------------------------

# Label encoder
with open("models/label_encoder.pkl", "rb") as f:
    le = pickle.load(f)

# Preprocessing config
with open("models/preprocess_config.pkl", "rb") as f:
    preprocess_config = pickle.load(f)

# Example: preprocess_config might look like
# {'scale_mode': 'bbox'} or {'scale_mode': 'max_dist'}

# -------------------------------
# 2. Start webcam
# -------------------------------

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    # Mirror view if needed (optional)
    # frame = cv2.flip(frame, 1)
    
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = hands.process(rgb_frame)
    
    if result.multi_hand_landmarks:
        for hand_landmarks in result.multi_hand_landmarks:
            mp_draw.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)
            
            # Extract raw landmarks
            landmarks = []
            for lm in hand_landmarks.landmark:
                landmarks.extend([lm.x, lm.y, lm.z])
            
            # -------------------------------
            # 3. Apply SAME preprocessing as training
            # -------------------------------
            landmarks = normalize_landmarks(landmarks, scale_mode=preprocess_config['scale_mode'])
            
            # Convert to numpy array and reshape
            X_input = np.array(landmarks).reshape(1, -1)
            
            # Optional: check min/max
            print("Post-preprocess min/max:", X_input.min(), X_input.max())
            
            # -------------------------------
            # 4. Predict
            # -------------------------------
            pred_probs = model.predict(X_input, verbose=0)
            pred_class = np.argmax(pred_probs)
            letter = le.inverse_transform([pred_class])[0]
            
            # Display
            cv2.putText(frame, f'Prediction: {letter}', (10, 50), 
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    
    cv2.imshow("FSL Webcam Test", frame)
    
    if cv2.waitKey(1) & 0xFF == 27:  # press Esc to exit
        break

cap.release()
cv2.destroyAllWindows()


In [ ]:
print(preprocess_config)


{'scale_mode': 'bbox'}


In [ ]:
pred_probs = model.predict(X_input, verbose=0)
print(pred_probs)


[[2.2156286e-08 1.8653269e-15 2.1656290e-06 3.1174847e-18 2.3757984e-08
  2.2147441e-13 2.2801875e-14 1.8505620e-17 1.2468723e-08 1.1361414e-12
  1.5890092e-18 9.5005205e-08 2.5879405e-09 2.2788160e-12 2.1686806e-07
  4.0728822e-03 9.9592304e-01 4.3810877e-19 7.5125277e-14 5.0089162e-19
  1.5219213e-19 1.6654014e-17 1.2790436e-16 6.0203375e-15 1.8341999e-10
  1.5536946e-06]]
